# Mutation-impact molecular dynamics workflow

This notebook performs replica-based structural ensemble analysis for a validated soluble-protein system. Membrane, ligand-bound, cofactor-dependent, metal-containing, and complex systems require their dedicated builders and parameterization branches; they are blocked from the generic solvent workflow rather than silently simplified.


In [6]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import time

import MDAnalysis as mda
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from MDAnalysis.analysis.rms import RMSD, RMSF
from openmm.app import PDBFile
from pdbfixer import PDBFixer

PDB_FILE = "uploads/ppox/3NKS_WT.pdb"
PROJECT_PREFIX = "PPOX_Study"
SYSTEM_TYPE = "soluble"  # soluble, membrane, complex, ligand_bound, metal, disordered
TARGET_CHAIN_ID = "A"
MUTATION_STR = "ALA-449-THR"
MUTATION_RES_ID = 449
MUTATION_FROM = "ALA"
MUTATION_TO = "THR"
APPLY_MUTATION = True
FORCE_FIELD = "charmm36m"
WATER_MODEL = "tip3p"
BOX_TYPE = "dodecahedron"
BOX_BUFFER_NM = 1.0
TEMPERATURE = 300.0
PRESSURE_BAR = 1.0
SALT_CONCENTRATION_M = 0.15
PH = 7.0
SIMULATION_TIME_NS = 100
N_REPLICAS = 3
GPU_ID = "0"
ANALYSIS_ENDPOINT = "structural_ensemble"
RUN_ALCHEMICAL_FREE_ENERGY = False
KEEP_WATER = True
REMOVE_HETEROGENS = False
LIGAND_RESNAMES = []
COFACTOR_RESNAMES = []
METAL_RESNAMES = []

if SYSTEM_TYPE != "soluble":
    raise ValueError("The generic branch is only for soluble proteins. Use CHARMM-GUI or a validated component-specific workflow for this system type.")
if REMOVE_HETEROGENS and (LIGAND_RESNAMES or COFACTOR_RESNAMES or METAL_RESNAMES):
    raise ValueError("Required ligands, cofactors, or metals cannot be removed.")
if N_REPLICAS < 3:
    raise ValueError("At least three independent replicas are required for comparative analysis.")
if ANALYSIS_ENDPOINT in {"folding_ddg", "binding_ddg"} and not RUN_ALCHEMICAL_FREE_ENERGY:
    raise ValueError("Folding/binding ddG requires an explicit alchemical free-energy workflow.")

NOTEBOOK_ROOT = Path.cwd().resolve()
PDB_PATH = (NOTEBOOK_ROOT / PDB_FILE).resolve()
WORK_DIR_BASE = NOTEBOOK_ROOT / "results"
SYSTEMS = ["WT", "Mutant"] if APPLY_MUTATION else ["WT"]
REPLICAS = [f"replica_{i:02d}" for i in range(1, N_REPLICAS + 1)]

print(f"Configured {SYSTEM_TYPE} system: {SYSTEMS}, {N_REPLICAS} replicas each")


Configured soluble system: ['WT', 'Mutant'], 3 replicas each


## 1. Structure preparation and mutation validation

Non-protein components are retained by default. Protonation, ligand, cofactor, and metal parameters must be reviewed before proceeding.


In [ ]:
def run_gmx(command, cwd, stdin_text=None):
    result = subprocess.run(command, cwd=cwd, input=stdin_text, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.returncode:
        print(result.stdout)
        raise RuntimeError(f"GROMACS failed: {' '.join(command)}")
    return result.stdout

def patch_mdp(path, updates):
    text = path.read_text()
    for key, value in updates.items():
        text, count = re.subn(rf"^{re.escape(key)}\s*=.*$", f"{key} = {value}", text, flags=re.MULTILINE)
        if count != 1:
            raise ValueError(f"Expected one {key} entry in {path}, found {count}")
    path.write_text(text)

def read_xvg(path):
    rows = []
    for line in path.read_text().splitlines():
        if line and not line.startswith(("@", "#")):
            rows.append([float(value) for value in line.split()])
    return pd.DataFrame(rows)

for system in SYSTEMS:
    output_dir = WORK_DIR_BASE / f"{PROJECT_PREFIX}_{system}"
    output_dir.mkdir(parents=True, exist_ok=True)
    fixer = PDBFixer(filename=str(PDB_PATH))
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    if REMOVE_HETEROGENS:
        fixer.removeHeterogens(keepWater=KEEP_WATER)
    if system == "Mutant":
        chains = [chain.id for chain in fixer.topology.chains()]
        if TARGET_CHAIN_ID not in chains:
            raise ValueError(f"Missing target chain {TARGET_CHAIN_ID}; available: {chains}")
        matches = [residue for chain in fixer.topology.chains() if chain.id == TARGET_CHAIN_ID for residue in chain.residues() if residue.id == str(MUTATION_RES_ID)]
        if len(matches) != 1 or matches[0].name != MUTATION_FROM:
            raise ValueError(f"Expected {MUTATION_FROM} at {TARGET_CHAIN_ID}:{MUTATION_RES_ID}")
        fixer.applyMutations([MUTATION_STR], TARGET_CHAIN_ID)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    with open(output_dir / "prepared.pdb", "w") as handle:
        PDBFile.writeFile(fixer.topology, fixer.positions, handle)
    print(f"Prepared {system}: {output_dir / 'prepared.pdb'}")


## 2. System building, equilibration, and independent production replicas

This branch uses matched CHARMM36m/TIP3P solvent, does not use `-maxwarn`, and uses named GROMACS groups rather than numeric selections. Membrane systems must use a membrane builder with semi-isotropic pressure coupling.


In [ ]:
for system in SYSTEMS:
    source_dir = WORK_DIR_BASE / f"{PROJECT_PREFIX}_{system}"
    for replica in REPLICAS:
        run_dir = source_dir / replica
        run_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_dir / "prepared.pdb", run_dir / "prepared.pdb")
        shutil.copytree(NOTEBOOK_ROOT / "mdp_templates", run_dir / "mdp", dirs_exist_ok=True)
        run_gmx(["gmx", "pdb2gmx", "-f", "prepared.pdb", "-o", "protein.gro", "-p", "topol.top", "-i", "posre.itp", "-ff", FORCE_FIELD, "-water", WATER_MODEL, "-ignh", "-ter"], run_dir, "1\n0\n")
        run_gmx(["gmx", "editconf", "-f", "protein.gro", "-o", "box.gro", "-c", "-d", str(BOX_BUFFER_NM), "-bt", BOX_TYPE], run_dir)
        run_gmx(["gmx", "solvate", "-cp", "box.gro", "-cs", "tip3p.gro", "-o", "solvated.gro", "-p", "topol.top"], run_dir)
        run_gmx(["gmx", "grompp", "-f", "mdp/em.mdp", "-c", "solvated.gro", "-r", "solvated.gro", "-p", "topol.top", "-o", "ions.tpr"], run_dir)
        run_gmx(["gmx", "genion", "-s", "ions.tpr", "-o", "solv_ions.gro", "-p", "topol.top", "-pname", "NA", "-nname", "CL", "-neutral", "-conc", str(SALT_CONCENTRATION_M)], run_dir, "SOL\n")

        patch_mdp(run_dir / "mdp/nvt.mdp", {"ref_t": f"{TEMPERATURE} {TEMPERATURE}"})
        patch_mdp(run_dir / "mdp/npt.mdp", {"ref_t": f"{TEMPERATURE} {TEMPERATURE}", "ref_p": str(PRESSURE_BAR), "refcoord-scaling": "all"})
        run_gmx(["gmx", "grompp", "-f", "mdp/em.mdp", "-c", "solv_ions.gro", "-r", "solv_ions.gro", "-p", "topol.top", "-o", "em.tpr"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "em", "-gpu_id", GPU_ID, "-nb", "gpu"], run_dir)
        run_gmx(["gmx", "grompp", "-f", "mdp/nvt.mdp", "-c", "em.gro", "-r", "em.gro", "-p", "topol.top", "-o", "nvt.tpr"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "nvt", "-gpu_id", GPU_ID, "-nb", "gpu"], run_dir)
        run_gmx(["gmx", "grompp", "-f", "mdp/npt.mdp", "-c", "nvt.gro", "-r", "nvt.gro", "-p", "topol.top", "-o", "npt.tpr", "-t", "nvt.cpt"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "npt", "-gpu_id", GPU_ID, "-nb", "gpu"], run_dir)

        prod_steps = int(SIMULATION_TIME_NS * 1000 / 0.002)
        patch_mdp(run_dir / "mdp/md.mdp", {"nsteps": prod_steps, "ref_t": str(TEMPERATURE), "tc-grps": "System", "tau_t": "1.0", "pcoupl": "Parrinello-Rahman", "tau_p": "5.0", "refcoord-scaling": "all"})
        run_gmx(["gmx", "grompp", "-f", "mdp/md.mdp", "-c", "npt.gro", "-r", "npt.gro", "-p", "topol.top", "-o", "md.tpr", "-t", "npt.cpt"], run_dir)
        run_gmx(["gmx", "mdrun", "-deffnm", "md", "-gpu_id", GPU_ID, "-nb", "gpu", "-pme", "gpu", "-bonded", "gpu", "-update", "gpu"], run_dir)
        run_gmx(["gmx", "trjconv", "-s", "md.tpr", "-f", "md.xtc", "-o", "md_center.xtc", "-pbc", "mol", "-center"], run_dir, "Protein\nSystem\n")
        run_gmx(["gmx", "trjconv", "-s", "md.tpr", "-f", "md_center.xtc", "-o", "md_fit.xtc", "-fit", "rot+trans"], run_dir, "Backbone\nSystem\n")
        print(f"Completed {system}/{replica}")


## 3. Replica-aware analysis and interpretation gate

Reported quantities are structural ensemble descriptors. RMSD/RMSF/SASA/Rg alone are not folding or binding free energies.


In [ ]:
records = []
for system in SYSTEMS:
    for replica in REPLICAS:
        run_dir = WORK_DIR_BASE / f"{PROJECT_PREFIX}_{system}" / replica
        universe = mda.Universe(str(run_dir / "md.gro"), str(run_dir / "md_fit.xtc"))
        backbone = universe.select_atoms("protein and backbone")
        rmsd = RMSD(universe, select="protein and backbone").run().results.rmsd
        rmsf = RMSF(backbone).run().results.rmsf / 10.0
        tail = rmsd[rmsd[:, 1] >= SIMULATION_TIME_NS * 500.0, 2].mean() / 10.0
        mutation_mask = (backbone.resids >= MUTATION_RES_ID - 5) & (backbone.resids <= MUTATION_RES_ID + 5)
        records.append({"System": system, "Replica": replica, "tail_rmsd_nm": tail, "global_rmsf_nm": rmsf.mean(), "mutation_window_rmsf_nm": rmsf[mutation_mask].mean()})
summary = pd.DataFrame(records)
if summary.groupby("System")["Replica"].nunique().min() < N_REPLICAS:
    raise ValueError("Not all requested replicas completed.")
summary_stats = summary.groupby("System").agg(["mean", "std", "count"])
display(summary_stats)
summary.to_csv(WORK_DIR_BASE / f"{PROJECT_PREFIX}_replica_summary.csv", index=False)
print("Interpretation gate passed: report replica means and uncertainty.")


## Methodological boundary

For folding stability, calculate a mutation free-energy difference using a validated alchemical thermodynamic cycle, such as pmx/FEP/TI with lambda-window overlap and independent convergence checks. For ligand binding, include both apo and complex legs. For membranes, ligands, cofactors, metals, or partner complexes, parameterize and validate every retained component before production.
